# Strict Video Protocol Benchmark

Production benchmark using the strict cached-prompt video propagation protocol. The default run is a broad smoke test: up to 1 valid cached-prompt case per dataset, all five models, and all three prompt modes (`box`, `mask`, `mixed`). Cases are ordered 2D first, then 3D, outputs are mirrored under `by_model/<model>/`, and TensorBoard summary tables update only when a dataset is completed for a model.

## 1. Setup And Config

In [ ]:
import os
import sys
import time
import importlib.util
from collections import OrderedDict
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display, Markdown


def find_helper_path():
    candidates = [
        Path.cwd() / 'strict_video_protocol_lib.py',
        Path.cwd() / 'notebooks' / 'strict_video_protocol_lib.py',
        Path('/RWKV-MedSAM2/notebooks/strict_video_protocol_lib.py'),
        Path('/workspace/RWKV-MedSAM2/notebooks/strict_video_protocol_lib.py'),
        Path('/data/RWKV-MedSAM2/notebooks/strict_video_protocol_lib.py'),
        Path('/data/rwkv_medsam2/notebooks/strict_video_protocol_lib.py'),
        Path('/data/jrbonill/RWKV-MedSAM2/notebooks/strict_video_protocol_lib.py'),
        Path('/data/jrbonill/rwkv_medsam2/notebooks/strict_video_protocol_lib.py'),
        Path('E:/dev/RWKV-MedSAM2/notebooks/strict_video_protocol_lib.py'),
        Path.home() / 'notebooks' / 'strict_video_protocol_lib.py',
        Path.home() / 'strict_video_protocol_lib.py',
    ]
    for helper in candidates:
        helper = helper.expanduser().resolve()
        if helper.is_file():
            return helper
    raise FileNotFoundError('Could not find strict_video_protocol_lib.py next to this notebook, in ~/notebooks, or in the RWKV-MedSAM2 repo')


HELPER_PATH = find_helper_path()
print('Helper path:', HELPER_PATH)

spec = importlib.util.spec_from_file_location('strict_video_protocol_lib', HELPER_PATH)
svp = importlib.util.module_from_spec(spec)
sys.modules['strict_video_protocol_lib'] = svp
spec.loader.exec_module(svp)
svp.OrderedDict = OrderedDict  # Backward-compatible fix for older helper copies used from /root.
svp.set_determinism()

print('REPO_ROOT:', svp.REPO_ROOT)
print('DEVICE:', svp.DEVICE)

BENCHMARK_MODELS = ['rwkv_medsam2_distill', 'rwkv_medsam2_nodistill', 'sam2_1_base', 'oxford_medical_sam2', 'uoft_medsam2']
BENCHMARK_PROMPT_MODES = ['box', 'mask', 'mixed']
STRICT_PROTOCOLS = ['validation_forward']

SMOKE_RUN = False
SMOKE_CASES_PER_DATASET = 1
SMOKE_CASE_SELECTION = 'per_dataset'
SMOKE_PREFER_SEQUENCE_CASES = True
SMOKE_DATASETS = None
SMOKE_MAX_SCAN_PER_DATASET = 250

MAX_TEST_CASES = None
BENCHMARK_CASE_INDICES = []
FAIL_FAST = False
RESUME_FROM_DIR = Path('/data/rwkv_medsam2_model_comparisons/strict_video_protocol_20260625_034426')
RESUME_COMPLETED_ROWS = True

THRESHOLDS = svp.THRESHOLDS
PRIMARY_THRESHOLD = svp.PRIMARY_THRESHOLD
COMPUTE_HD95_2D = True
COMPUTE_HD95_2D_FOR_3D_CASES = False
COMPUTE_HD95_3D = False
BENCHMARK_TENSORBOARD = True
PARTIAL_SAVE_EVERY_ROWS = 100 if SMOKE_RUN else 25000
STATUS_UPDATE_EVERY_ROWS = 25
STATUS_UPDATE_EVERY_SEC = 60.0
CHECKPOINT_FLUSH_EVERY_ROWS = 1
ORDER_2D_BEFORE_3D = True
SAVE_MODEL_SEPARATED_OUTPUTS = True
CACHE_CASES_CPU = True
CACHE_PROMPT_PLANS_CPU = True
WRITE_MODEL_OUTPUTS_ON_PARTIAL = False
REUSE_PROMPT_MODE_STATE = False  # Disable after kernel deaths during native CUDA inference; keep metric vectorization enabled.
REUSE_PROMPT_MODE_STATE_MODELS = ['rwkv_medsam2_distill']
VISUAL_TOP_N_DATASETS = None

RUN_STAMP = time.strftime('strict_video_protocol_%Y%m%d_%H%M%S')
OUTPUT_BASE = Path('/data/rwkv_medsam2_model_comparisons')
if not OUTPUT_BASE.exists():
    OUTPUT_BASE = svp.REPO_ROOT / 'strict_video_benchmark_outputs'
OUTPUT_DIR = OUTPUT_BASE / RUN_STAMP
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)

## 2. Load Test Split

In [ ]:
train_loader, val_loader, test_loader = svp.load_benchmark_loaders()
test_dataset = getattr(test_loader, 'dataset', test_loader)
print('Test cases after exclusion:', len(test_dataset))

## 3. Run Benchmark

In [ ]:
BENCHMARK_OUTPUTS = svp.run_strict_video_benchmark(
    test_dataset,
    output_dir=OUTPUT_DIR,
    benchmark_models=BENCHMARK_MODELS,
    prompt_modes=BENCHMARK_PROMPT_MODES,
    protocols=STRICT_PROTOCOLS,
    smoke_run=SMOKE_RUN,
    smoke_cases_per_dataset=SMOKE_CASES_PER_DATASET,
    smoke_case_selection=SMOKE_CASE_SELECTION,
    smoke_prefer_sequence_cases=SMOKE_PREFER_SEQUENCE_CASES,
    smoke_datasets=SMOKE_DATASETS,
    smoke_max_scan_per_dataset=SMOKE_MAX_SCAN_PER_DATASET,
    max_test_cases=MAX_TEST_CASES,
    benchmark_case_indices=BENCHMARK_CASE_INDICES,
    thresholds=THRESHOLDS,
    primary_threshold=PRIMARY_THRESHOLD,
    compute_hd95_2d=COMPUTE_HD95_2D,
    compute_hd95_2d_for_3d_cases=COMPUTE_HD95_2D_FOR_3D_CASES,
    compute_hd95_3d=COMPUTE_HD95_3D,
    tensorboard=BENCHMARK_TENSORBOARD,
    partial_save_every_rows=PARTIAL_SAVE_EVERY_ROWS,
    status_update_every_rows=STATUS_UPDATE_EVERY_ROWS,
    status_update_every_sec=STATUS_UPDATE_EVERY_SEC,
    checkpoint_flush_every_rows=CHECKPOINT_FLUSH_EVERY_ROWS,
    order_2d_before_3d=ORDER_2D_BEFORE_3D,
    save_model_separated_outputs=SAVE_MODEL_SEPARATED_OUTPUTS,
    fail_fast=FAIL_FAST,
    visual_top_n_datasets=VISUAL_TOP_N_DATASETS,
    resume_from_dir=RESUME_FROM_DIR,
    resume_completed_rows=RESUME_COMPLETED_ROWS,
    cache_cases_cpu=CACHE_CASES_CPU,
    cache_prompt_plans_cpu=CACHE_PROMPT_PLANS_CPU,
    write_model_outputs_on_partial=WRITE_MODEL_OUTPUTS_ON_PARTIAL,
    reuse_prompt_mode_state=REUSE_PROMPT_MODE_STATE,
    reuse_prompt_mode_state_models=REUSE_PROMPT_MODE_STATE_MODELS,
)
print('Benchmark complete:', BENCHMARK_OUTPUTS['output_dir'])

## 4. Inspect Outputs

In [ ]:
print('Output dir:', BENCHMARK_OUTPUTS['output_dir'])

def display_case_selection_preview(df):
    display(Markdown('### case_selection'))
    if df.empty:
        display(df)
        return
    selected = df[df.get('status', pd.Series(index=df.index, dtype=str)).astype(str).eq('selected')].copy()
    if selected.empty:
        selected = df.copy()
    group_cols = [c for c in ['order_phase', 'order_dim', 'status'] if c in selected.columns]
    if group_cols:
        display(Markdown('#### Selection phase summary'))
        display(selected.groupby(group_cols, dropna=False).size().reset_index(name='rows'))
    preview_cols = [c for c in ['benchmark_order', 'dataset_name', 'dataset_index', 'status', 'order_phase', 'order_dim', 'sequence_length', 'foreground_frame_count', 'cached_prompt_frames'] if c in selected.columns]
    phase_col = 'order_phase' if 'order_phase' in selected.columns else None
    if phase_col:
        for phase in selected[phase_col].dropna().astype(str).unique():
            display(Markdown(f'#### First selected rows: {phase}'))
            display(selected[selected[phase_col].astype(str).eq(phase)].loc[:, preview_cols].head(12))
    else:
        display(Markdown('#### First selected rows'))
        display(selected.loc[:, preview_cols].head(25))
    display(Markdown('#### Last selected rows'))
    display(selected.loc[:, preview_cols].tail(25))

for key in ['api_probe', 'case_selection', 'status_summary', 'summary_dataset_macro', 'best_overall', 'memory_summary', 'efficiency']:
    df = BENCHMARK_OUTPUTS.get(key)
    if isinstance(df, pd.DataFrame):
        if key == 'case_selection':
            display_case_selection_preview(df)
        else:
            display(Markdown(f'### {key}'))
            display(df.head(50))

print('Core files:')
for path in [
    OUTPUT_DIR / 'per_case_results.csv',
    OUTPUT_DIR / 'prompt_audit.csv',
    OUTPUT_DIR / 'per_frame_metrics.csv',
    OUTPUT_DIR / 'threshold_metrics_long.csv',
    OUTPUT_DIR / 'memory_measurements.csv',
    OUTPUT_DIR / 'shareable_summary.md',
    OUTPUT_DIR / 'detailed_summary.md',
    OUTPUT_DIR / 'summary_packet.zip',
]:
    print(' -', path, 'exists=', path.exists())